# 04d - Model Comparison

**Objective**: Compare all forecasting models and select the best

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
print('✓ Libraries imported')

In [ ]:
# Load all forecastsprophet_fc = pd.read_csv('../models/trained/prophet_forecast.csv', parse_dates=['ds'])sarima_fc = pd.read_csv('../models/trained/sarima_forecast.csv', parse_dates=['date'])xgb_fc = pd.read_csv('../models/trained/xgboost_forecast.csv', parse_dates=['date'])try:    actual = pd.read_csv('../data/clean/maize_features.csv')except FileNotFoundError:    actual = pd.read_csv('data/clean/maize_features.csv')print('✓ Forecasts loaded')

In [ ]:
# Create comparison dataframe
test_actual = actual.iloc[-12:][['date', 'price']]
test_actual = test_actual.reset_index(drop=True)
prophet_test = prophet_fc.tail(12)[['ds', 'yhat']].reset_index(drop=True)
prophet_test.columns = ['date', 'prophet_forecast']
comparison = test_actual.merge(prophet_test, on='date')
comparison = comparison.merge(sarima_fc[['date', 'forecast']].rename(columns={'forecast': 'sarima_forecast'}), on='date')
comparison = comparison.merge(xgb_fc[['date', 'forecast']].rename(columns={'forecast': 'xgb_forecast'}), on='date')
print(comparison)

In [ ]:
# Calculate metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
metrics = {}
for model in ['prophet_forecast', 'sarima_forecast', 'xgb_forecast']:
    mae = mean_absolute_error(comparison['price'], comparison[model])
    rmse = np.sqrt(mean_squared_error(comparison['price'], comparison[model]))
    mape = np.mean(np.abs((comparison['price'] - comparison[model]) / comparison['price'])) * 100
    metrics[model] = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}
metrics_df = pd.DataFrame(metrics).T
print('\n=== MODEL COMPARISON ===')
print(metrics_df.round(2))

In [ ]:
from pathlib import PathPath('../models/trained').mkdir(parents=True, exist_ok=True)# Save comparisonmetrics_df.to_csv('../models/trained/model_comparison.csv')comparison.to_csv('../models/trained/forecasts_comparison.csv', index=False)print('✓ Comparison saved')


In [ ]:
# Best model
best_model = metrics_df['MAPE'].idxmin()
print(f'\n🏆 Best Model: {best_model}')
print(f'MAPE: {metrics_df.loc[best_model, "MAPE"]:.2f}%')